# 03 — The routing strategies, side by side

Nine dispatch policies, one interface, identical demand. This is the notebook
that shows *what was actually compared* and how.

Every policy implements one method:

```python
DispatchPolicy.plan(state: BusState, n_stops: int) -> list[str]
```

It is handed a snapshot — where the bus is, who is aboard, who is waiting, how
many seats are free, what time it is — and returns the stops to visit next.
Nothing else differs between the arms below: same map, same seeds, same demand,
same bus. Swapping the policy is the only variable.

Runtime: about three minutes at `"quick"`, fifteen at `"full"`.

In [ ]:
# --- Bootstrap ------------------------------------------------------------
# This notebook imports the project as `src.…`, which requires the repository
# root on the import path. `./jupyter-run.sh` puts it there; opening the file
# any other way does not. This cell finds the root by walking up from the
# working directory, so the notebook runs wherever it is launched from and
# whatever folder it sits in.
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Seeded runs only reproduce when Python's string hashing is pinned: several
# policies break ties with min() over a set, whose iteration order is
# randomised per process otherwise. The simulation refuses to run unpinned;
# this says so now rather than after a long cell.
if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print("WARNING: PYTHONHASHSEED is not pinned, so seeded runs will be "
          "refused.\n         Launch with ./jupyter-run.sh, or set "
          "PYTHONHASHSEED=0.")

print(f"repository root: {ROOT}")

In [ ]:
import os

# How much work this notebook does. Every notebook in this folder has this
# switch and honours the AUTOBUS_SCALE environment variable, so nothing here
# needs editing to run it bigger or smaller.
#
#   "quick"  the default: minutes, same conclusions, wider error bars
#   "full"   the sample sizes behind the figures in the submitted documents
#   "smoke"  a tiny end-to-end check that the notebook runs at all
#
# The outputs saved in this file were produced at "full".
SCALE = os.environ.get("AUTOBUS_SCALE", "quick")
print(f"running at SCALE = {SCALE!r}")

In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

## The nine, and where each came from

They were built in this order, and the order is the argument — each one exists
because of what the previous one failed to do.

| Policy | Idea | Family |
|---|---|---|
| `FixedRoutePolicy` | Drive a fixed cycle through every stop, forever | baseline |
| `SkipStopFixedRoutePolicy` | The same cycle, skipping stops nobody wants | fixed route |
| `DeviatedFixedRoutePolicy` | Skip-stop, plus bounded detours to nearby demand | fixed route |
| `NearestUnservedPolicy` | Greedily drive to the nearest demanded stop | greedy |
| `MarginalCostFilterPolicy` | Greedy, ignoring pickups that cost too much to reach | greedy |
| `RideBoundedGreedyPolicy` | Greedy, forced to deliver when a rider is over budget | greedy |
| `ConstrainedInsertionPolicy` | Cheapest insertion into the route, rejecting the infeasible | insertion |
| `WindowInsertionPolicy` | Insertion under a promised pickup deadline (**L0**) | insertion |
| `ReorderingInsertionPolicy` | L0, plus repair of the order already planned (**L1**) | insertion |

The last two come from the dial-a-ride literature: window insertion is the
**ADARTW** heuristic (Jaw, Odoni, Psaraftis & Wilson, 1986), and the repair
pass is the classical relocate neighbourhood that insertion construction is
normally paired with.

In [ ]:
# The campus map: the world behind every synthetic result in this project.
# stop_density=0.5 is load-bearing -- the generator's default of 0.15 gives a
# 6-stop map on this seed, on which every policy ties. See notebook 01.
CAMPUS = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0), seed=42,
              stop_density=0.5)

In [ ]:
from src.algorithm.ConstrainedInsertionPolicy import ConstrainedInsertionPolicy
from src.algorithm.DeviatedFixedRoutePolicy import DeviatedFixedRoutePolicy
from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.MarginalCostFilterPolicy import MarginalCostFilterPolicy
from src.algorithm.NearestUnservedPolicy import NearestUnservedPolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.RideBoundedGreedyPolicy import RideBoundedGreedyPolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy

# The service window is set to passenger patience, which is the recommended
# setting and the subject of notebook 04.
WINDOW = 900.0
PATIENCE = 900.0

POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Skip-stop", lambda g: SkipStopFixedRoutePolicy(g)),
    ("Deviated", lambda g: DeviatedFixedRoutePolicy(g)),
    ("Nearest", lambda g: NearestUnservedPolicy(g)),
    ("Marginal cost", lambda g: MarginalCostFilterPolicy(g)),
    ("Ride bounded", lambda g: RideBoundedGreedyPolicy(g)),
    ("Constrained", lambda g: ConstrainedInsertionPolicy(g)),
    ("Window (L0)", lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW)),
    ("Reordering (L1)",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]

TRIALS = {"smoke": 2, "quick": 20, "full": 100}[SCALE]
REQUESTS = {"smoke": 40, "quick": 200, "full": 200}[SCALE]

print(f"{len(POLICIES)} policies x {TRIALS} trials of {REQUESTS} requests")

## How the comparison is made fair

`compare_policies` runs trial *i* of every policy with **the same demand
seed**. Every arm therefore faces an identical sequence of passengers asking
for identical journeys at identical times, and the only thing that differs is
what the bus did about it.

That pairing is what makes small differences readable: the seed-to-seed
variation, which is large, cancels out of a per-seed comparison.

In [ ]:
from src.experiment import summarize_comparison
from src.map.graph import PlanarStreetGraph

comparison = summarize_comparison(
    POLICIES,
    graph=PlanarStreetGraph.generate(**CAMPUS),
    n_trials=TRIALS,
    base_seed=0,
    max_requests=REQUESTS,
    demand_interval_s=60.0,
    capacity=40,
    max_wait_s=PATIENCE,
)

display(Markdown(comparison.markdown))

Each cell is the mean over trials, with the 95% confidence
interval half-width as a percentage of that mean in brackets. A figure marked
(±2%) is solid; one marked (±40%) is a direction, not a number.

In [ ]:
from src.experiment import plot_kpi_bars

plot_kpi_bars(
    comparison.results,
    ["served_fraction", "mean_wait_time", "mean_ride_time", "throughput"],
    title=f"Nine policies, {TRIALS} paired trials",
)
plt.show()

## Reading it

**Fixed route is the baseline, and it is not a straw man.** It serves the
fewest passengers, but its ride time is a property of the cycle rather than of
demand — it cannot be surprised. Everything else has to beat it while giving
that up.

**Skip-stop dominates plain fixed route** — same cycle order, so the same
bounded worst-case wait, but it does not drive to stops nobody wants. Plain
fixed route is kept only as the control.

**Greedy wins wait and loses ride, catastrophically.** Driving to the nearest
demanded stop means there is always a closer pickup, so onboard passengers get
dragged around the map. Look at the ride distribution below, not the mean.

**Marginal cost ≈ Nearest.** The filter is inert: it removes far pickups,
which greedy was not going to choose anyway. A constraint only does something
if it forbids what the policy actually wanted to do.

**The insertion policies win because they change the visit *order*.** Every
other idea here bolts a constraint onto somebody else's ordering — greedy's or
the cycle's — and so has to trade coverage against ride. Insertion optimises
the order itself and does not face that trade.

In [ ]:
from src.experiment import plot_wait_distribution

plot_wait_distribution(comparison.results)
plt.show()

In [ ]:
import numpy as np

# The frontier the project spent a month on: coverage against the ride tail.
figure, axis = plt.subplots(figsize=(7.6, 5.0))
for label, trials in comparison.results.items():
    served = np.mean([t.kpi.served_fraction for t in trials]) * 100
    worst_ride = np.mean([max(t.ride_times) if t.ride_times else 0
                          for t in trials])
    axis.scatter(worst_ride, served, s=70, zorder=3)
    axis.annotate(label, (worst_ride, served), fontsize=8,
                  xytext=(6, 4), textcoords="offset points")
axis.set_xlabel("worst ride time in a run (s)  ->  worse")
axis.set_ylabel("served (%)  ->  better")
axis.set_title("Coverage against the ride tail")
axis.grid(alpha=0.3)
figure.tight_layout()

The top-left corner is where a good policy lives: high
coverage, no passenger driven around for hours. The greedy family sits far to
the right, the fixed family to the left but low, and the insertion policies
occupy the corner.

## Does the bus actually keep up?

The per-tick trace separates a policy that serves everyone slowly from one
that serves some people and abandons the rest.

In [ ]:
from src.experiment import compare_policies, plot_occupancy

# A three-way trace on a single seed, with the tick history kept. Keeping it
# for every trial above would hold one snapshot per tick per trial in memory.
traced = compare_policies(
    [POLICIES[0], POLICIES[1], POLICIES[-1]],
    graph=PlanarStreetGraph.generate(**CAMPUS),
    n_trials=1, base_seed=0, max_requests=REQUESTS, demand_interval_s=60.0,
    capacity=40, max_wait_s=PATIENCE, keep_tick_history=True,
)
plot_occupancy(traced)
plt.show()

## Watching two policies drive the same demand

Set `ANIMATE = True` to render side-by-side animations of the fixed route and
the recommended policy on the same seed. It is off by default only because the
rendered frames make the saved notebook large — the code is the same one-liner
notebook 00 uses.

In [ ]:
ANIMATE = False

if ANIMATE:
    from tqdm.auto import tqdm

    from src.experiment import run_trial, show_animations

    watched = {
        label: run_trial(
            policy_factory=factory,
            graph=PlanarStreetGraph.generate(**CAMPUS),
            seed=0, max_requests=30, demand_interval_s=60.0, capacity=40,
            max_wait_s=PATIENCE, record_animation=True, silent=True)
        for label, factory in tqdm([POLICIES[0], POLICIES[-1]],
                                   desc="recording", unit="policy")
    }
    show_animations(watched)
else:
    print("ANIMATE = False -- set it True to render the side-by-side run")

## What this notebook establishes

1. **The ranking**, on this map at these settings: the insertion policies lead
   on coverage, the fixed family owns the ride tail, and greedy is a warning.
2. **That the comparison is paired**, which is what makes the differences
   readable at all.
3. **That two reasonable-sounding ideas failed** — `Ride bounded` and
   `Deviated` both collapsed toward an endpoint of the frontier instead of
   improving on it. Notebook 04 sweeps their knobs and shows *why*, which is a
   better result than quietly dropping them.

What it does **not** establish is that the ranking holds anywhere but here.
One map, one demand rate, one bus size. **Notebook 05** repeats it across 25
maps and asks whether the answer is stable; **notebook 06** finds the
conditions under which it inverts.